# 21 BestTime Feasibility

Small-credit feasibility test for BestTime using a handful of major Istanbul landmarks.

Official reference used for the request shape:
- `POST https://besttime.app/api/v1/forecasts`
- requires `api_key_private`, `venue_name`, `venue_address`
- source: [BestTime API Reference](https://documentation.besttime.app/app/)

This notebook is intentionally conservative so it does not become a costly branch.

In [ ]:
import ast
import os
import time

import pandas as pd
import requests

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


In [ ]:
INPUT_PATH = "../data/processed/otm_pois_base_final.csv"
OUTPUT_PATH = "../data/processed/besttime_feasibility_top8.csv"

BESTTIME_FORECAST_URL = "https://besttime.app/api/v1/forecasts"
BESTTIME_API_KEY = os.getenv("BESTTIME_API_KEY")
if not BESTTIME_API_KEY:
    BESTTIME_API_KEY = input("Paste BestTime private API key: ").strip()

print("Key loaded:", bool(BESTTIME_API_KEY))

poi_df = pd.read_csv(INPUT_PATH)
poi_df.shape


## Pick a very small landmark test set

In [ ]:
TARGET_NAMES = [
    "Hagia Sophia",
    "The Blue Mosque",
    "Topkapı Palace",
    "Dolmabahçe Palace",
    "Galata Tower",
    "Spice Bazaar",
    "Süleymaniye Mosque",
    "Eyüp Sultan Mosque",
    "Rumeli Fortress",
    "Fatih Mosque",
    "Mihrimah Sultan Mosque",
]

test_df = poi_df[poi_df["display_name_en"].isin(TARGET_NAMES)].copy()
missing_targets = [name for name in TARGET_NAMES if name not in set(test_df["display_name_en"])]
print("Matched targets:", len(test_df))
print("Missing targets:", missing_targets)
test_df[["otm_xid", "display_name_en", "query_area", "address_raw"]]


## Helpers

In [ ]:
def parse_dict_like(value):
    if pd.isna(value):
        return {}
    if isinstance(value, dict):
        return value
    text = str(value).strip()
    if not text:
        return {}
    try:
        return ast.literal_eval(text)
    except Exception:
        return {}


def build_besttime_address(row):
    address = parse_dict_like(row.get("address_raw"))
    parts = []
    for key in ["road", "pedestrian", "suburb", "town", "city", "state"]:
        value = address.get(key)
        if value and str(value) not in parts:
            parts.append(str(value))
    parts.append("Istanbul")
    parts.append("Turkey")
    return ", ".join(dict.fromkeys(parts))


def call_besttime_forecast(venue_name, venue_address):
    params = {
        "api_key_private": BESTTIME_API_KEY,
        "venue_name": venue_name,
        "venue_address": venue_address,
    }
    response = requests.post(BESTTIME_FORECAST_URL, params=params, timeout=30)
    data = response.json()
    if response.status_code != 200:
        debug = {
            "status_code": response.status_code,
            "text": response.text[:500],
            "url": response.url,
        }
        print("BestTime error debug:", debug)
        return data
    return data


def summarize_analysis(data):
    analysis = data.get("analysis") or []
    if isinstance(analysis, dict):
        analysis = [analysis]

    num_days = len(analysis)
    any_busy_hours = False
    any_day_info = False

    for day in analysis:
        if not isinstance(day, dict):
            continue
        if day.get("busy_hours"):
            any_busy_hours = True
        if day.get("day_info"):
            any_day_info = True

    return {
        "besttime_has_analysis": num_days > 0,
        "besttime_num_days": num_days,
        "besttime_any_busy_hours": any_busy_hours,
        "besttime_any_day_info": any_day_info,
    }


## Run a tiny feasibility test

In [ ]:
rows = []

for _, row in test_df.iterrows():
    venue_name = row["display_name_en"]
    venue_address = build_besttime_address(row)

    record = {
        "otm_xid": row["otm_xid"],
        "display_name_en": venue_name,
        "query_area": row["query_area"],
        "besttime_query_name": venue_name,
        "besttime_query_address": venue_address,
        "besttime_found": False,
        "besttime_status": "",
        "besttime_venue_name": None,
        "besttime_venue_address": None,
        "besttime_venue_id": None,
        "besttime_has_analysis": False,
        "besttime_num_days": 0,
        "besttime_any_busy_hours": False,
        "besttime_any_day_info": False,
        "besttime_notes": "",
        "review_correct_place": "",
        "review_would_use": "",
    }

    try:
        print(f"Checking: {venue_name}")
        data = call_besttime_forecast(venue_name, venue_address)
        record["besttime_status"] = data.get("status", "")
        venue_info = data.get("venue_info", {})
        record["besttime_venue_name"] = venue_info.get("venue_name")
        record["besttime_venue_address"] = venue_info.get("venue_address")
        record["besttime_venue_id"] = venue_info.get("venue_id")
        record.update(summarize_analysis(data))
        record["besttime_found"] = bool(venue_info.get("venue_name") or record["besttime_has_analysis"])
        if data.get("message"):
            record["besttime_notes"] = data.get("message", "")
    except Exception as exc:
        record["besttime_status"] = f"error: {type(exc).__name__}"
        record["besttime_notes"] = str(exc)

    rows.append(record)
    time.sleep(1.0)

besttime_df = pd.DataFrame(rows)
besttime_df


## Save and inspect

In [ ]:
besttime_df.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH, besttime_df.shape)

besttime_df[[
    "display_name_en",
    "besttime_found",
    "besttime_status",
    "besttime_venue_name",
    "besttime_has_analysis",
    "besttime_num_days",
    "besttime_any_busy_hours",
    "besttime_any_day_info",
    "review_correct_place",
    "review_would_use",
]].to_string(index=False)


## Notes

- This notebook is intentionally small because BestTime forecast calls can cost credits.
- The goal is only to answer whether BestTime is worth integrating further.
- If the returned venue names/addresses look wrong, mark that in the review columns instead of forcing the integration.